# Testing Ollama Locally

Download [Ollama](https://ollama.com/) and run it terminal with: ollama run llama3.1:8b. 

Note: Takes longer than running anything on Colab with a T4. 

In [ ]:
from langchain_ollama import ChatOllama
import pandas as pd

from tqdm import tqdm
import time
import re
from langchain_core.prompts import ChatPromptTemplate
from sklearn.metrics import classification_report

In [ ]:
# Initialize the model
# You can adjust 'temperature' for creativity vs. logic
model = ChatOllama(model="llama3.1:8b", temperature=0.2, top_p=0.9)

# Run a simple test
prompt = "Who are you?"
response = model.invoke(prompt)

print(response)

## Vanilla Prompting Setup

In [ ]:
df = pd.read_csv("../data/sampled_50_dataset.csv")

In [ ]:
system_msg = "You are a classifier. Output ONLY '0' or '1'. No explanation."
user_msg = "Classify this text (0=Human, 1=AI):\n\n{text200}"

prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("user", user_msg)
])

chain = prompt | model

# 3. Preparation
texts = df["text200"].tolist()
results = []
start = time.time()

batch_size = 16

for i in tqdm(range(0, len(texts), batch_size), desc="T4 Batch Processing"):
    batch_input = [{"text200": t} for t in texts[i : i + batch_size]]

    # .batch() is much faster on T4 than .invoke() in a loop
    responses = chain.batch(batch_input)

    for res in responses:
        # Clean the output string
        clean_content = res.content.strip()

        # Look for the last digit if the model was chatty
        matches = re.findall(r'\b[01]\b', clean_content)
        if matches:
            # We take the last match as the final answer
            results.append(int(matches[-1]))
        else:
            results.append(-1) # Error flag

df["ai_opinion_200_va"] = results
print(f"Total time: {time.time() - start:.2f}s")

In [ ]:
y_true = df['ai_target']
y_pred = df['ai_opinion_200_va']

print(classification_report(y_true, y_pred))